In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Örnek İHA Teknik & Operasyon Kılavuzu (Chunks / Bilgi Parçaları)
# Gerçek hayatta 500 sayfalık bir PDF'i belirli karakter/kelime sınırlarıyla (örneğin 300 kelimelik parçalar ve 50 kelimelik örtüşme/overlap ile) böleriz.
dokuman_parcalari = [
    "BÖLÜM 1 - MOTOR VE İTKİ SİSTEMİ: KIZILELMA-X İHA platformunda ana itki sistemi maksimum 115 derece egzoz gazı sıcaklığında sürekli çalışabilir. Sıcaklık 120 dereceyi aşarsa otomatik güç kısma (derating) modu devreye girer.",
    "BÖLÜM 2 - HABERLEŞME VE DATALINK: Yer kontrol istasyonu (YKİ) ile görüş hattı (LOS) haberleşmesi C-Bant üzerinden sağlanır. Sinyal kaybı 15 saniyeyi aşarsa İHA otomatik olarak 'Eve Dönüş' (RTH - Return to Home) prosedürünü başlatır.",
    "BÖLÜM 3 - BATARYA VE GÜÇ YÖNETİMİ: Ana aviyonik batarya voltajı 22.4V altına düştüğünde kritik düşük voltaj uyarısı verilir. Bu durumda faydalı yük (kamera/radar) gücü kesilerek yalnızca uçuş kontrol bilgisayarına enerji aktarılır.",
    "BÖLÜM 4 - İNİŞ TAKIMLARI VE FRENLEME: Hidrolik iniş takımları yerden 150 metre irtifada açılır. İniş sonrası karbon-seramik frenlerin soğuma süresi minimum 20 dakikadır; bu süre dolmadan yeni kalkış izni verilmez.",
    "BÖLÜM 5 - BUZLANMA ÖNLEME SİSTEMİ: Kanat hücum kenarlarındaki elektro-termal buz çözücü sistem, dış hava sıcaklığı +2 derecenin altına düştüğünde ve nem oranı %80'i geçtiğinde otonom olarak aktifleşir."
]

# 2. Türkçe destekli çok dilli Embedding modelini yüklüyoruz
print("Embedding modeli yükleniyor...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 3. Doküman parçalarını sayısal vektörlere (Embedding) dönüştürüyoruz
dokuman_vektorleri = embed_model.encode(dokuman_parcalari)

print(f"Toplam parça sayısı: {dokuman_vektorleri.shape[0]}")
print(f"Her bir parçanın vektör boyutu (kaç sayıdan oluştuğu): {dokuman_vektorleri.shape[1]}")
print(f"İlk parçanın ilk 5 sayısı: {dokuman_vektorleri[0][:5]}")

# 4. FAISS Vektör Veri Tabanı oluşturma ve vektörleri içine ekleme
vektor_boyutu = dokuman_vektorleri.shape[1]  # 384 boyut
index = faiss.IndexFlatL2(vektor_boyutu)    # Öklid mesafesi tabanlı arama indeksi
index.add(np.array(dokuman_vektorleri).astype("float32"))

print("Vektör veri tabanı (FAISS) başarıyla oluşturuldu!")

Embedding modeli yükleniyor...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Toplam parça sayısı: 5
Her bir parçanın vektör boyutu (kaç sayıdan oluştuğu): 384
İlk parçanın ilk 5 sayısı: [0.00277909 0.28100023 0.0117034  0.08953051 0.01050485]
Vektör veri tabanı (FAISS) başarıyla oluşturuldu!


In [ ]:
def dokuman_ara(soru, top_k=2):
  # 1. Soruyu da aynı embedding modeliyle 384 boyutlu vektöre çeviriyoruz
  soru_vektoru = embed_model.encode([soru]).astype("float32")

  # 2. FAISS indeksinde en yakın 'top_k' parçayı arıyoruz
  # mesafeler: L2 (Öklid) uzaklığı (0'a ne kadar yakınsa o kadar benzer)
  # indeksler: eşleşen doküman parçalarının sıra numarası
  mesafeler, indeksler = index.search(soru_vektoru, k=top_k)

  print(f"SORU: '{soru}'")
  print("-" * 60)
  for sira, (idx, mesafe) in enumerate(zip(indeksler[0], mesafeler[0]), 1):
    print(f"Eşleşme #{sira} | Parça #{idx} | L2 Uzaklığı: {mesafe:.4f}")
    print(f"İçerik: {dokuman_parcalari[idx]}\n")


# Test 1: Kelimeler birebir geçmiyor ("pil" ve "kamera" kılavuzda farklı geçiyor)
dokuman_ara("Pil azalırsa kamera çalışmaya devam eder mi?", top_k=2)

print("=" * 70 + "\n")

# Test 2: "Bağlantı koparsa" yerine "sinyal kaybı" eşleşecek mi görelim
dokuman_ara("Yerle bağlantı koparsa uçak ne yapar?", top_k=1)

SORU: 'Pil azalırsa kamera çalışmaya devam eder mi?'
------------------------------------------------------------
Eşleşme #1 | Parça #2 | L2 Uzaklığı: 14.8559
İçerik: BÖLÜM 3 - BATARYA VE GÜÇ YÖNETİMİ: Ana aviyonik batarya voltajı 22.4V altına düştüğünde kritik düşük voltaj uyarısı verilir. Bu durumda faydalı yük (kamera/radar) gücü kesilerek yalnızca uçuş kontrol bilgisayarına enerji aktarılır.

Eşleşme #2 | Parça #1 | L2 Uzaklığı: 15.3981
İçerik: BÖLÜM 2 - HABERLEŞME VE DATALINK: Yer kontrol istasyonu (YKİ) ile görüş hattı (LOS) haberleşmesi C-Bant üzerinden sağlanır. Sinyal kaybı 15 saniyeyi aşarsa İHA otomatik olarak 'Eve Dönüş' (RTH - Return to Home) prosedürünü başlatır.


SORU: 'Yerle bağlantı koparsa uçak ne yapar?'
------------------------------------------------------------
Eşleşme #1 | Parça #2 | L2 Uzaklığı: 15.0082
İçerik: BÖLÜM 3 - BATARYA VE GÜÇ YÖNETİMİ: Ana aviyonik batarya voltajı 22.4V altına düştüğünde kritik düşük voltaj uyarısı verilir. Bu durumda faydalı yük (kam

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Hafif, hızlı ve Türkçe anlama/yazma kabiliyeti güçlü bir LLM yükleyelim
model_adi = "Qwen/Qwen2.5-1.5B-Instruct"

print("Dil modeli (LLM) yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(model_adi)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_adi,
    torch_dtype=torch.float16,  # GPU belleğinden tasarruf için yarı hassasiyet
    device_map="auto",  # Otomatik olarak T4 GPU'ya yerleştir
)


def rag_cevapla(soru, top_k=2):
  # 1. RETRIEVAL: Soruyu vektörleştir ve FAISS'ten en yakın parçaları çek
  soru_vektoru = embed_model.encode([soru]).astype("float32")
  mesafeler, indeksler = index.search(soru_vektoru, k=top_k)

  secilen_parcalar = [dokuman_parcalari[i] for i in indeksler[0]]
  baglam = "\n\n".join(secilen_parcalar)

  # 2. AUGMENTATION & GENERATION: Bağlamı prompt'a koyup LLM'den cevap üret
  sistem_mesaji = (
      "Sen bir İHA teknik doküman asistanısın. Aşağıdaki 'KILAVUZ BİLGİSİ' "
      "bölümünü kullanarak soruyu kısa, net ve doğru şekilde yanıtla. "
      "Bilgi kılavuzda açıkça yer almıyorsa 'Kılavuzda bu bilgi yer almıyor' de, kesinlikle uydurma."
  )
  kullanici_mesaji = f"KILAVUZ BİLGİSİ:\n{baglam}\n\nSORU: {soru}"

  mesajlar = [
      {"role": "system", "content": sistem_mesaji},
      {"role": "user", "content": kullanici_mesaji},
  ]

  metin_girdisi = tokenizer.apply_chat_template(
      mesajlar, tokenize=False, add_generation_prompt=True
  )
  girdi_tensörleri = tokenizer([metin_girdisi], return_tensors="pt").to(
      llm_model.device
  )

  with torch.no_grad():
    cikti_idler = llm_model.generate(
        **girdi_tensörleri,
        max_new_tokens=150,  # En fazla 150 yeni token üret
        temperature=0.2,  # Düşük temperature = daha tutarlı, daha az yaratıcı/uydurgan cevap
        do_sample=True,
    )

  # Sadece yeni üretilen cevap kısmını ayıkla
  yeni_tokenler = cikti_idler[0][girdi_tensörleri.input_ids.shape[1] :]
  cevap = tokenizer.decode(yeni_tokenler, skip_special_tokens=True)

  print(f"SORU: {soru}")
  print(f"CEVAP: {cevap}\n")
  print("Kullanılan Kaynak Parçalar:")
  for i, p in enumerate(secilen_parcalar, 1):
    print(f"  [{i}] {p}")
  print("-" * 70)


# Test edelim!
rag_cevapla("Pil azalırsa kamera çalışmaya devam eder mi?", top_k=2)
rag_cevapla("İHA'nın yakıt deposu kaç litre?", top_k=2)  # Kılavuzda olmayan soru!

Dil modeli (LLM) yükleniyor...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

SORU: Pil azalırsa kamera çalışmaya devam eder mi?
CEVAP: Kılavuzda bu bilgi yer almıyor.

Kullanılan Kaynak Parçalar:
  [1] BÖLÜM 3 - BATARYA VE GÜÇ YÖNETİMİ: Ana aviyonik batarya voltajı 22.4V altına düştüğünde kritik düşük voltaj uyarısı verilir. Bu durumda faydalı yük (kamera/radar) gücü kesilerek yalnızca uçuş kontrol bilgisayarına enerji aktarılır.
  [2] BÖLÜM 2 - HABERLEŞME VE DATALINK: Yer kontrol istasyonu (YKİ) ile görüş hattı (LOS) haberleşmesi C-Bant üzerinden sağlanır. Sinyal kaybı 15 saniyeyi aşarsa İHA otomatik olarak 'Eve Dönüş' (RTH - Return to Home) prosedürünü başlatır.
----------------------------------------------------------------------
SORU: İHA'nın yakıt deposu kaç litre?
CEVAP: Bu bilgiye göre, İHA'nın yakıt deposu bilgisi yoktur. Bu durumda "Kılavuzda bu bilgi yer almıyor" olarak yanıtlamak daha uygun olacaktır.

Kullanılan Kaynak Parçalar:
  [1] BÖLÜM 1 - MOTOR VE İTKİ SİSTEMİ: KIZILELMA-X İHA platformunda ana itki sistemi maksimum 115 derece egzoz gazı sıcak